⚠️ **Gemini Parse Error** — response could not be parsed as a valid notebook.
Raw output preserved below for manual recovery.

In [ ]:
{
  "cells": [
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "# Migration: Oracle ODI to Databricks Spark SQL\n",
        "**Source File:** target_odi_sql.txt\n",
        "**Conversion Date:** 2024-05-22\n",
        "**Description:** Conversion of file system commands, table truncation, and sequence management."
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "dbutils.widgets.text(\"DATASOURCE_NUM_ID\", \"\")\n",
        "dbutils.widgets.text(\"ETL_PROC_WID\", \"\")\n",
        "dbutils.widgets.text(\"ODI_SESS_NO\", \"\")"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## ETL Parameters"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "print(f\"Processing Session: {dbutils.widgets.get('ODI_SESS_NO')}\")"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## File Management (SCEN_TASK_NO {1})"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "# Converted OdiOSCommand: dir /a /b C:\\\ODI\\\employee > C:\\\ODI\\\employeesfile\\name.csv\n",
        "# In Databricks, local file system paths (C:\\\...) must be replaced with DBFS/Unity Catalog Volume paths.\n",
        "try:\n",
        "    files = dbutils.fs.ls(\"/mnt/odi/employee\")\n",
        "    file_names = \"\\n\".join([f.name for f in files])\n",
        "    dbutils.fs.put(\"/mnt/odi/employeesfile/name.csv\", file_names, overwrite=True)\n",
        "except Exception as e:\n",
        "    print(f\"File operation failed: {e}\")"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Truncate Log Table (SCEN_TASK_NO {2})"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "-- MAGIC %sql\n",
        "TRUNCATE TABLE workspace.odi_trg.log_table;"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Sequence Management (SCEN_TASK_NO {3})"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "-- MAGIC %sql\n",
        "-- Oracle sequences are converted to Delta Lake IDENTITY columns.\n",
        "-- Replacing standalone sequence restart with table-based identity restart.\n",
        "-- Replace <table_name> and <column_name> with the table using file_sequence.\n",
        "ALTER TABLE workspace.odi_trg.log_table ALTER COLUMN id RESTART WITH 1;"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Validation"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "-- MAGIC %sql\n",
        "SELECT COUNT(*) as log_count FROM workspace.odi_trg.log_table;"
      ]
    }
  ],
  "metadata": {
    "kernelspec": {
      "display_name": "Python 3",
      "language": "python",
      "name": "python3"
    },
    "language_info": {
      "codemirror_mode": {
        "name": "ipython",
        "version": 3
      },
      "file_extension": ".py",
      "mimetype": "text/x-python",
      "name": "python",
      "nbconvert_exporter": "python",
      "pygments_lexer": "ipython3",
      "version": "3.10.12"
    }
  },
  "nbformat": 4,
  "nbformat_minor": 5
}